# Duplicates and data quality · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 11. Duplicates and data quality

Find repeated records and validate IDs, ranges, and allowed labels.

Before you start: You can filter rows, convert values, and count missing entries.

Official reference: https://pandas.pydata.org/docs/user_guide/indexing.html#duplicate-data

### M11C01. Find an exact repeated record

`duplicated()` marks repeated rows. By default, the first occurrence is kept unmarked and later copies are marked True. With `keep=False`, every member of a repeated set is marked.

**Read:** Why use keep=False when investigating duplicates?

Your explanation: _write here_.

**Predict:** How many displayed rows and how many extra copies will we see?

Your explanation: _write here_.

**Explain:** Is a repeated name enough to identify a duplicate person?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many displayed rows and how many extra copies will we see?
df = raw_students()
result = df.loc[df.duplicated(keep=False), ["Student_ID", "Name", "Score"]]
print(result)
print("Extra copies:", df.duplicated().sum())

**Your turn:** Remove exact duplicate rows and reset the index. Store the result in `result`.

For the check, keep these result names: `result`.

**Hint:** Use `.drop_duplicates().reset_index(drop=True)`.

**Watch for this:** A duplicate record and a missing value are different problems.

In [ ]:
# Change this code to complete the task.
# Question: Remove exact duplicate rows and reset the index. Store the result in `result`.
df = raw_students()
result = df.loc[df.duplicated(keep=False), ["Student_ID", "Name", "Score"]]
print(result)
print("Extra copies:", df.duplicated().sum())

**Quick check:** What does duplicated() count by default here?

1. Every occurrence of a repeated row
2. Only repeated copies after the first
3. Rows with missing values

**Explain what you learned:** Inspect repeated records before removing them.

### M11C02. A repeated ID is not always an exact duplicate

`subset` lets you check only the columns that define a record key. Two rows can share an ID but disagree on a score. Keeping the last one is defensible only when a documented order says it is the latest record.

**Read:** Which field is being treated as the record key?

Your explanation: _write here_.

**Predict:** Are the two S001 rows exact duplicates?

Your explanation: _write here_.

**Explain:** Why would dropping one row without a rule be risky?

Your explanation: _write here_.



In [ ]:
# Predict before running: Are the two S001 rows exact duplicates?
df = pd.DataFrame({"Student_ID": ["S001", "S001", "S002"], "Updated": ["2026-10-01", "2026-10-03", "2026-10-02"], "Score": [80, 88, 72]})
result = df.loc[df.duplicated(subset=["Student_ID"], keep=False)]
print(result)
print("Exact duplicates:", df.duplicated().sum())

**Your turn:** Assume the later Updated date is the authoritative revision. Parse Updated, sort chronologically, and keep the latest row per Student_ID. Store the retained rows in result, sorted by Student_ID in ascending order.

For the check, keep these result names: `result`.

**Hint:** Date parsing is introduced in Module 13: use pd.to_datetime with format="%Y-%m-%d". Sort by Updated before drop_duplicates(subset="Student_ID", keep="last"), then sort the retained rows by Student_ID.

**Watch for this:** In a transaction table, repeated student IDs may be expected; define what one row represents.

In [ ]:
# Change this code to complete the task.
# Question: Assume the later Updated date is the authoritative revision. Parse Updated, sort chronologically, and keep the latest row per Student_ID. Store the retained rows in result, sorted by Student_ID in ascending order.
df = pd.DataFrame({"Student_ID": ["S001", "S001", "S002"], "Updated": ["2026-10-01", "2026-10-03", "2026-10-02"], "Score": [80, 88, 72]})
result = df.loc[df.duplicated(subset=["Student_ID"], keep=False)]
print(result)
print("Exact duplicates:", df.duplicated().sum())

**Quick check:** Does keep="last" automatically know the newest date?

1. Yes
2. No; the row order must first match the date rule
3. Only when the ID is numeric

**Explain what you learned:** Choose a key and a documented conflict rule.

### M11C03. Flag impossible values instead of hiding them

Data can have the right type and still be invalid. If Attendance is a percentage, its allowed range is 0 through 100. A Boolean mask can create a review table before you decide how to repair the source.

**Read:** What does the ~ symbol do to the range check?

Your explanation: _write here_.

**Predict:** Which record will be flagged?

Your explanation: _write here_.

**Explain:** Why should we not automatically change 105 to 100?

Your explanation: _write here_.



In [ ]:
# Predict before running: Which record will be flagged?
df = raw_students()
invalid = ~df["Attendance"].between(0, 100)
result = df.loc[invalid, ["Student_ID", "Attendance"]]
print(result)

**Your turn:** Preserve the source Attendance values in Attendance_raw. Convert Attendance to a nullable numeric column and set its out-of-range values to missing. Store the affected rows in result with columns Student_ID, Attendance_raw, and Attendance.

For the check, keep these result names: `result`.

**Hint:** Use a copy, `astype("Float64")`, and one `.loc[mask, column]` assignment.

**Watch for this:** The allowed range comes from the meaning of the variable, not from pandas.

In [ ]:
# Change this code to complete the task.
# Question: Preserve the source Attendance values in Attendance_raw. Convert Attendance to a nullable numeric column and set its out-of-range values to missing. Store the affected rows in result with columns Student_ID, Attendance_raw, and Attendance.
df = raw_students()
invalid = ~df["Attendance"].between(0, 100)
result = df.loc[invalid, ["Student_ID", "Attendance"]]
print(result)

**Quick check:** Is a numeric value always a valid measurement?

1. Yes
2. No, it must also satisfy the measurement rules
3. Only when it has a decimal point

**Explain what you learned:** Validate values against explicit domain rules.

### M11C04. Turn expectations into assertions

An assertion stops execution if an expectation is false. It is useful for classroom checks and analysis pipelines. Validate identity, required values, and ranges separately so a failure has a clear meaning.

**Read:** What happens if an assertion condition is False?

Your explanation: _write here_.

**Predict:** Will these checks pass for students()?

Your explanation: _write here_.

**Explain:** Why test missing IDs separately from uniqueness?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will these checks pass for students()?
df = students()
assert df["Student_ID"].notna().all(), "A student ID is missing"
assert df["Student_ID"].is_unique, "Student IDs must be unique here"
assert df["Score"].between(0, 100).all(), "Score outside 0-100"
result = {"rows": len(df), "valid_ids": True, "valid_scores": True}
print(result)

**Your turn:** Add checks requiring Campus to be RAK or Dubai and Attendance to be 0-100. Store the two Boolean results in a dictionary named result with keys campus_ok and attendance_ok.

For the check, keep these result names: `result`.

**Hint:** Use `.isin([...]).all()` for an allowed-label check.

**Watch for this:** Python can disable assert statements in optimized runs; formal applications may need explicit validation errors.

In [ ]:
# Change this code to complete the task.
# Question: Add checks requiring Campus to be RAK or Dubai and Attendance to be 0-100. Store the two Boolean results in a dictionary named result with keys campus_ok and attendance_ok.
df = students()
assert df["Student_ID"].notna().all(), "A student ID is missing"
assert df["Student_ID"].is_unique, "Student IDs must be unique here"
assert df["Score"].between(0, 100).all(), "Score outside 0-100"
result = {"rows": len(df), "valid_ids": True, "valid_scores": True}
print(result)

**Quick check:** For a Boolean Series with no missing values, what does .all() require?

1. At least one True
2. Every value to be True
3. Exactly twelve rows

**Explain what you learned:** Write data expectations as checks you can repeat.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates